![display relevant image here](path/url/to/image)
- Banner/header image

# Title
- Relevant to Data and Business Context

## Overview
- BLUF (Bottom Line Up Front)
- One paragraph summary of findings and analysis
- Frame your 'story'

## AI Integration Notes

Throughout this project, an AI assistant (Claude) was used as a collaborative debugging and design-review partner rather than as a code generator that wrote the analysis unprompted. Key uses:

- **Debugging PySpark logic:** e.g., diagnosing why a NULL-comparison filter was silently dropping a row I had just fixed, and explaining that comparisons against null in Spark return null rather than true/false.
- **Data quality investigation:** confirming hypotheses about data issues (e.g., verifying that repeated App entries were re-scraped snapshots rather than exact duplicates, by pulling and inspecting the actual rows).
- **Chart design review:** critiquing early visualization attempts (e.g., flagging a diverging rainbow color palette as a poor fit for single-scale count data, and identifying when a linear-scale scatter plot was unreadable due to outliers, prompting a switch to log scale).
- **Dashboard layout feedback:** suggestions on making legends/filters more compact (floating placement, dropdown filters) to preserve chart space.

All code was written and run by me in my own PySpark/Tableau environment; the AI did not have direct access to run code against this dataset. Suggestions were evaluated, tested, and in several cases modified or rejected (e.g., an initial suggestion to fix all 3 malformed rows was scoped down to fixing only the one row that could be corrected with confidence, dropping the other 2 as not worth the effort for 2 rows).

## Business Understanding
This analysis explores the Google Play Store dataset to help app developers and product stakeholders at a startup better understand what drives user engagement and monetization outcomes for mobile apps. These findings matter because a startup deciding on pricing strategy, category positioning, or feature investment needs data-backed guidance rather than guesswork, as every development and marketing dollar spent should be informed by what actually correlates with success in the market.

**Stakeholders:** Product managers, marketing teams, and business leadership deciding on app category, pricing model, and feature priorities.

**Questions this analysis aims to answer:**
1. Do free apps get more reviews relative to installs than paid apps (an engagement proxy)?
2. Which categories have the highest review-to-install ratio?
3. Does app size relate to engagement / do smaller, lighter apps get used/reviewed more?
4. Does price correlate with rating / are paid apps rated higher/lower than free ones?
5. Which categories have the most paid apps, and how does their engagement compare to free apps in the same category?
6. Using Price × Installs as a rough revenue-potential proxy, which categories look the most lucrative for paid apps?

**Note on data limitations:** this dataset doesn't include real revenue, daily active users, or retention data, so "monetization" here is necessarily approximated using Price and Installs rather than actual revenue figures.

## Data Understanding
**Source:** Google Play Store dataset (10,841 rows, 13 columns), containing app-level metadata: App name, Category, Rating, Reviews, Size, Installs, Type (Free/Paid), Price, Content Rating, Genres, Last Updated, Current Ver, and Android Ver.

**Relevant columns for this analysis:** Category, Rating, Reviews, Installs, Type, Price, and Size are most relevant to the engagement/monetization questions above. Genres, Current Ver, and Android Ver are less central but retained for context.

**Data quality issues identified during EDA:**

1. **3 structurally malformed rows**, two rows have unescaped special characters in the App name field, causing values to spill into the wrong columns (Category, Rating, Reviews). One row ("Life Made WI-Fi Touchscreen Photo Frame") is missing its Category value entirely, shifting every subsequent column one position to the left.

2. **1,474 missing ratings (~13.6% of rows)**, represented as the string `"NaN"` rather than a true null value — a standard `.isNull()` check does not catch these, so this had to be checked for explicitly.

3. **483 repeated App entries**, investigation showed these are not exact duplicate rows, but the same app scraped at different points in time (differing Reviews counts, and in some cases differing Category values, e.g. ROBLOX listed as both "GAME" and "FAMILY" across scrapes). Since this analysis is not time-series focused, only the snapshot with the highest Reviews count per app will be retained.

4. **Nearly all columns are typed as strings**, including ones that should be numeric, like Rating, Reviews, Installs (contains commas and "+" symbols), and Price (contains "$" symbols and the string "Free"). Size mixes units ("19M", "1.2G") and includes the non-numeric value "Varies with device". These will require type conversion and standardization in the Data Preparation phase.

In [54]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

In [55]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Play Store Analysis") \
    .getOrCreate()
if(spark.getActiveSession()):
    print('Session Active')
else: 
    print('Session inactive, Activating now..')
    spark = SparkSession.builder \
    .appName("Play Store Analysis") \
    .getOrCreate() 
    #just as redundancy / if for some reason the initial 3 lines dont run.

df = spark.read.csv("google_play_store_dataset.csv", header=True, inferSchema=True)
print("Row count:", df.count())
df.printSchema()

Session Active
Row count: 10841
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



In [56]:
# EDA Code Here - Create New Cells As Needed
# Statistical Analysis
df.describe().show()

# Distinct value counts for categorical columns
for col in ["Category", "Type", "Content Rating", "Genres"]:
    print(col, "-> distinct values:", df.select(col).distinct().count())

+-------+--------------------+-------------+-----------+-----------------+------------------+------------------+-----+------------------+--------------+------+-----------------+-------------+------------------+
|summary|                 App|     Category|     Rating|          Reviews|              Size|          Installs| Type|             Price|Content Rating|Genres|     Last Updated|  Current Ver|       Android Ver|
+-------+--------------------+-------------+-----------+-----------------+------------------+------------------+-----+------------------+--------------+------+-----------------+-------------+------------------+
|  count|               10841|        10841|      10841|            10841|             10841|             10841|10841|             10841|         10840| 10841|            10841|        10840|             10840|
|   mean|                NULL|          1.9|        NaN|444225.1924709356|              NULL| 2.866666666666667|  NaN|               0.0|          NULL|  NU

In [57]:
# Null value check per column
for col in df.columns:
    null_count = df.filter(F.col(col).isNull()).count()
    print(col, "-> nulls:", null_count)

# Duplicate check
print("Total rows:", df.count())
print("Distinct rows:", df.distinct().count())
print("Duplicate rows:", df.count() - df.distinct().count())

App -> nulls: 0
Category -> nulls: 0
Rating -> nulls: 0
Reviews -> nulls: 0
Size -> nulls: 0
Installs -> nulls: 0
Type -> nulls: 0
Price -> nulls: 0
Content Rating -> nulls: 1
Genres -> nulls: 0
Last Updated -> nulls: 0
Current Ver -> nulls: 1
Android Ver -> nulls: 1
Total rows: 10841
Distinct rows: 10358
Duplicate rows: 483


In [58]:
# Check for rows where Category looks invalid (not a real category)
df.groupBy("Category").count().orderBy("count").show(40, truncate=False)

+-------------------+-----+
|Category           |count|
+-------------------+-----+
|Face               |1    |
|1.9                |1    |
| traffic jams      |1    |
|BEAUTY             |53   |
|COMICS             |60   |
|PARENTING          |60   |
|EVENTS             |64   |
|ART_AND_DESIGN     |65   |
|WEATHER            |82   |
|AUTO_AND_VEHICLES  |85   |
|LIBRARIES_AND_DEMO |85   |
|HOUSE_AND_HOME     |88   |
|FOOD_AND_DRINK     |127  |
|MAPS_AND_NAVIGATION|136  |
|ENTERTAINMENT      |149  |
|EDUCATION          |156  |
|VIDEO_PLAYERS      |175  |
|BOOKS_AND_REFERENCE|231  |
|DATING             |234  |
|TRAVEL_AND_LOCAL   |258  |
|SHOPPING           |260  |
|NEWS_AND_MAGAZINES |283  |
|SOCIAL             |295  |
|PHOTOGRAPHY        |335  |
|HEALTH_AND_FITNESS |340  |
|FINANCE            |366  |
|LIFESTYLE          |382  |
|SPORTS             |384  |
|COMMUNICATION      |387  |
|PERSONALIZATION    |392  |
|PRODUCTIVITY       |424  |
|BUSINESS           |460  |
|MEDICAL            

In [ ]:
# Check for missing values in the "Content Rating" column and display duplicate app entries
df.filter(F.col("Content Rating").isNull()).show(truncate=False)
df.groupBy("App").count().filter(F.col("count") > 1).orderBy(F.desc("count")).show(10)

+---------------------------------------+--------+------+-------+------+--------+----+--------+--------------+-----------------+------------+-----------+-----------+
|App                                    |Category|Rating|Reviews|Size  |Installs|Type|Price   |Content Rating|Genres           |Last Updated|Current Ver|Android Ver|
+---------------------------------------+--------+------+-------+------+--------+----+--------+--------------+-----------------+------------+-----------+-----------+
|Life Made WI-Fi Touchscreen Photo Frame|1.9     |19    |3.0M   |1,000+|Free    |0   |Everyone|NULL          |February 11, 2018|1.0.19      |4.0 and up |NULL       |
+---------------------------------------+--------+------+-------+------+--------+----+--------+--------------+-----------------+------------+-----------+-----------+

+--------------------+-----+
|                 App|count|
+--------------------+-----+
|              ROBLOX|    9|
|CBS Sports App - ...|    8|
|                ESPN|  

In [ ]:
# counting NaNs in ratings
df.filter(F.col("Rating") == "NaN").count()


1474

In [ ]:
# checking dupes of Roblox for cleanup
df.filter(F.col("App") == "ROBLOX").show(truncate=False)

+------+--------+------+-------+----+------------+----+-----+--------------+----------------------------+-------------+------------+-----------+
|App   |Category|Rating|Reviews|Size|Installs    |Type|Price|Content Rating|Genres                      |Last Updated |Current Ver |Android Ver|
+------+--------+------+-------+----+------------+----+-----+--------------+----------------------------+-------------+------------+-----------+
|ROBLOX|GAME    |4.5   |4447388|67M |100,000,000+|Free|0    |Everyone 10+  |Adventure;Action & Adventure|July 31, 2018|2.347.225742|4.1 and up |
|ROBLOX|GAME    |4.5   |4447346|67M |100,000,000+|Free|0    |Everyone 10+  |Adventure;Action & Adventure|July 31, 2018|2.347.225742|4.1 and up |
|ROBLOX|GAME    |4.5   |4448791|67M |100,000,000+|Free|0    |Everyone 10+  |Adventure;Action & Adventure|July 31, 2018|2.347.225742|4.1 and up |
|ROBLOX|GAME    |4.5   |4449882|67M |100,000,000+|Free|0    |Everyone 10+  |Adventure;Action & Adventure|July 31, 2018|2.347.22574

## Data Preparation



In [67]:
# Data Prep Code Here
# Step 1: drop the 2 unfixable malformed rows FIRST, before touching Category
df_step1 = df.filter(~F.col("Category").isin("Face", " traffic jams"))
print("After dropping 2 broken rows:", df_step1.count())  # should be 10839

# Step 2: now fix the "Life Made" row on the reduced dataframe
df_fixed = df_step1.withColumn(
    "Category_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", None).otherwise(F.col("Category"))
).withColumn(
    "Rating_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "1.9").otherwise(F.col("Rating"))
).withColumn(
    "Reviews_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "19").otherwise(F.col("Reviews"))
).withColumn(
    "Size_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "3.0M").otherwise(F.col("Size"))
).withColumn(
    "Installs_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "1,000+").otherwise(F.col("Installs"))
).withColumn(
    "Type_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "Free").otherwise(F.col("Type"))
).withColumn(
    "Price_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "0").otherwise(F.col("Price"))
).withColumn(
    "ContentRating_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "Everyone").otherwise(F.col("Content Rating"))
).withColumn(
    "Genres_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", None).otherwise(F.col("Genres"))
).withColumn(
    "LastUpdated_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "February 11, 2018").otherwise(F.col("Last Updated"))
).withColumn(
    "CurrentVer_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "1.0.19").otherwise(F.col("Current Ver"))
).withColumn(
    "AndroidVer_fixed",
    F.when(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame", "4.0 and up").otherwise(F.col("Android Ver"))
)

df_fixed = df_fixed.drop("Category", "Rating", "Reviews", "Size", "Installs", "Type", "Price", "Content Rating",
                          "Genres", "Last Updated", "Current Ver", "Android Ver")
for old, new in [("Category","Category_fixed"),("Rating","Rating_fixed"),("Reviews","Reviews_fixed"),
                 ("Size","Size_fixed"),("Installs","Installs_fixed"),("Type","Type_fixed"),
                 ("Price","Price_fixed"),("Content Rating","ContentRating_fixed"),
                 ("Genres","Genres_fixed"),("Last Updated","LastUpdated_fixed"),
                 ("Current Ver","CurrentVer_fixed"),("Android Ver","AndroidVer_fixed")]:
    df_fixed = df_fixed.withColumnRenamed(new, old)

print("Final row count:", df_fixed.count())  # should still be 10839


After dropping 2 broken rows: 10839
Final row count: 10839


In [68]:
# verification check for malformed row fix
df_fixed.filter(F.col("App") == "Life Made WI-Fi Touchscreen Photo Frame").show(truncate=False)

+---------------------------------------+--------+------+-------+----+--------+----+-----+--------------+------+-----------------+-----------+-----------+
|App                                    |Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated     |Current Ver|Android Ver|
+---------------------------------------+--------+------+-------+----+--------+----+-----+--------------+------+-----------------+-----------+-----------+
|Life Made WI-Fi Touchscreen Photo Frame|NULL    |1.9   |19     |3.0M|1,000+  |Free|0    |Everyone      |NULL  |February 11, 2018|1.0.19     |4.0 and up |
+---------------------------------------+--------+------+-------+----+--------+----+-----+--------------+------+-----------------+-----------+-----------+



In [70]:
# dedupe phase
from pyspark.sql import Window

window = Window.partitionBy("App").orderBy(F.desc(F.col("Reviews").cast("long")))
df_deduped = df_fixed.withColumn("row_num", F.row_number().over(window)) \
                      .filter(F.col("row_num") == 1) \
                      .drop("row_num")

print("Before dedup:", df_fixed.count())
print("After dedup:", df_deduped.count())

Before dedup: 10839
After dedup: 9658


In [ ]:
# sanity check
df_deduped.filter(F.col("App") == "ROBLOX").show(truncate=False)

+------+--------+------+-------+----+------------+----+-----+--------------+----------------------------+-------------+------------+-----------+
|App   |Category|Rating|Reviews|Size|Installs    |Type|Price|Content Rating|Genres                      |Last Updated |Current Ver |Android Ver|
+------+--------+------+-------+----+------------+----+-----+--------------+----------------------------+-------------+------------+-----------+
|ROBLOX|FAMILY  |4.5   |4450890|67M |100,000,000+|Free|0    |Everyone 10+  |Adventure;Action & Adventure|July 31, 2018|2.347.225742|4.1 and up |
+------+--------+------+-------+----+------------+----+-----+--------------+----------------------------+-------------+------------+-----------+



In [74]:
# type conversion
df_typed = df_deduped \
    .withColumn("Rating", F.when(F.col("Rating") == "NaN", None).otherwise(F.col("Rating")).cast("float")) \
    .withColumn("Reviews", F.col("Reviews").cast("long")) \
    .withColumn("Installs", F.regexp_replace(F.col("Installs"), "[+,]", "").cast("long")) \
    .withColumn("Price", F.regexp_replace(F.col("Price"), "[$]", "").cast("float"))

df_typed.select("Rating", "Reviews", "Installs", "Price").show(10)
df_typed.printSchema()

+------+-------+--------+-----+
|Rating|Reviews|Installs|Price|
+------+-------+--------+-----+
|  NULL|     27|     500|  0.0|
|   4.5|      2|     100|  0.0|
|   4.6|   4925| 1000000|  0.0|
|   4.4|   3031|  500000|  0.0|
|  NULL|      1|      10| 5.99|
|   4.5|  40467| 1000000|  0.0|
|   3.5|    115|   10000|  0.0|
|   4.5|    259|   10000|  0.0|
|   4.7|    573|   10000|  0.0|
|   4.4|     27|     100|  0.0|
+------+-------+--------+-----+
only showing top 10 rows
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



In [ ]:
# checking size to get kilobyte instead of mega or Varies
df_typed.filter(~F.col("Size").rlike("^[0-9.]+M$")) \
    .select("Size").distinct().show(20, truncate=False)

+----+
|Size|
+----+
|970k|
|691k|
|913k|
|458k|
|957k|
|245k|
|317k|
|246k|
|749k|
|193k|
|82k |
|992k|
|48k |
|716k|
|176k|
|34k |
|73k |
|485k|
|44k |
|27k |
+----+
only showing top 20 rows


In [76]:
# further checks for Mega/Kilo/Varies
size_numeric = F.regexp_extract(F.col("Size"), "([0-9.]+)", 1).cast("float")
size_converted = F.when(F.col("Size").endswith("M"), size_numeric * 1024 * 1024) \
                   .when(F.col("Size").endswith("k"), size_numeric * 1024) \
                   .otherwise(None)

df_typed = df_typed.withColumn("Size_bytes", size_converted)

df_typed.select("Size", "Size_bytes").distinct().show(10)
df_typed.printSchema()

+-----+-----------+
| Size| Size_bytes|
+-----+-----------+
| 6.6M|  6920601.5|
|1020k|  1044480.0|
|  13M|1.3631488E7|
|  16M|1.6777216E7|
| 219k|   224256.0|
| 437k|   447488.0|
| 8.4M|  8808038.0|
| 6.7M|  7025459.0|
|  19M|1.9922944E7|
|  17k|    17408.0|
+-----+-----------+
only showing top 10 rows
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: float (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)
 |-- Size_bytes: double (nullable = true)



In [77]:
# Validation checks
print("Total rows:", df_typed.count())
print("Nulls after cleaning:")
for col in ["Rating", "Reviews", "Installs", "Price", "Size_bytes"]:
    print(" ", col, "->", df_typed.filter(F.col(col).isNull()).count())

# Confirm numeric ranges make sense
df_typed.select("Rating", "Price", "Installs").summary("min", "max", "mean").show()

Total rows: 9658
Nulls after cleaning:
  Rating -> 1463
  Reviews -> 0
  Installs -> 0
  Price -> 0
  Size_bytes -> 1227
+-------+-----------------+------------------+------------------+
|summary|           Rating|             Price|          Installs|
+-------+-----------------+------------------+------------------+
|    min|              1.0|               0.0|                 0|
|    max|              5.0|             400.0|        1000000000|
|   mean|4.172959116990425|1.0973441506745432|7798356.5362393875|
+-------+-----------------+------------------+------------------+



In [78]:
# Save cleaned data file for Tableau usage
df_typed.toPandas().to_csv("cleaned_play_store_data.csv", index=False)
print("Export complete.")

c:\Users\zache\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Export complete.


In [79]:
if(spark.getActiveSession()):
    print('Closing session..')
    spark.stop()
    print('Session is closed.')
else: print('Session is already closed.') #in case its ran again

Closing session..
Session is closed.


### [Tableau](https://public.tableau.com/app/profile/zachery.croft/viz/AD-FND1-SUMLAB-PT1-VIZ/PricingMonetization?publish=yes)

## Data Analysis

Six core visualizations were built in Tableau to answer the business questions defined above, each addressing a specific engagement or monetization question:

1. **Average Rating by Category**, identifies which app categories are best-rated overall, giving a baseline view of category-level quality/satisfaction.
2. **Reviews vs. Installs by Category** (log-log scatter), tests whether higher-reach categories (more installs) also see proportionally higher engagement (more reviews). A clear, consistent upward trend was found across nearly the full range of categories.
3. **Average Rating vs. Variance of Price, split by Free/Paid**, tests whether inconsistent pricing within a category relates to how that category is rated. No meaningful relationship was found; ratings cluster tightly (roughly 4.0–4.5) regardless of price variance, for both Free and Paid apps.
4. **Estimated Revenue Potential (Price × Installs) by Category**, a rough proxy for which categories have the greatest monetization potential under a paid pricing model. FAMILY, LIFESTYLE, and GAME lead by a wide margin.
5. **Average App Size vs. Average Engagement (Reviews), colored by Rating**, tests whether larger apps see more or less engagement. No clear relationship was found; engagement appears driven by factors other than app size.
6. **Average Reviews: Free vs. Paid, by Category**, directly compares engagement between free and paid apps within each category. Free apps substantially outperform paid apps on average reviews in every single category, often by one or two orders of magnitude.

These six visualizations were assembled into a three-page interactive Tableau dashboard (Overview, Engagement Analysis, Pricing & Monetization), with cross-page navigation and a Category filter allowing stakeholders to explore individual or multiple categories across all charts.

## Conclusion

This analysis set out to understand what drives engagement and monetization outcomes for apps on the Google Play Store, in order to support pricing, category, and feature decisions for a startup. Several clear, actionable findings emerged: engagement (reviews) scales consistently with reach (installs) across nearly all categories, suggesting that install growth reliably translates into user engagement rather than one growing at the expense of the other. Free apps dramatically outperform paid apps on engagement in every category, reinforcing that a freemium or ad-supported model is likely to generate far more user interaction than an upfront-paid model, even in categories where paid apps do exist and succeed.

Equally important were the negative findings: neither price consistency (variance) nor app size showed a meaningful relationship with rating or engagement, suggesting that developers likely shouldn't expect gains in user satisfaction or engagement purely from pricing strategy or app size decisions alone. Theres other factors, not captured in this dataset (such as UX quality, marketing, or content specifics), are more likely to be driving these outcomes.

**Limitations:** This dataset lacks true revenue, retention, or daily-active-user data, so "monetization" was necessarily approximated using Price × Installs rather than actual financial performance. Additionally, roughly 13.6% of apps had no rating, and a small number of structurally corrupted rows required correction or removal during cleaning; these limitations are documented in the Data Understanding and Data Preparation sections above.